### Save aggregated GSEA results

In [ ]:
from gseapy import dotplot, barplot
import gseapy as gp
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np


for perturb_seq_name in ['K562_GWPS', 'K562_Essential','RPE1_Essential','HepG2_Essential','Jurkat_Essential','RPE1_Southard']:
    print(perturb_seq_name)
    prior_names = ['Marbach', 'MsigDB', 'DepMap', 'StringDB', 'Periscope', 'Untangled', 'scGPT', 'genePT']
    column_names = [x+'_'+perturb_seq_name for x in prior_names]

    all_enriched_terms = []
    all_gsea_results = []

    for col in prior_names:
        
        tmp = pd.read_csv(f'/home/jhpark/page_reproducibility/03_prior_perturbseq_compare/intermediate/mrnaspace_{perturb_seq_name}_{col}.csv')
        tmp = tmp[tmp["FDR q-val"]<0.001].sort_values("NES")[::-1]
        all_gsea_results.append(tmp)
        #sns.barplot(tmp, y="Term", x="NES", color="black", orient="y")
        #plt.grid(False)
        all_enriched_terms += tmp['Term'].tolist()

    all_enriched_terms = list(set(all_enriched_terms))

    print('No. of enriched terms:', len(all_enriched_terms))

    plot_gsea_data = []
    geneset_metadata = []
    for terms in all_enriched_terms:
        geneset_metadata.append(terms.split('__')[0])
        tmp = []
        for col_id in range(8):
            gsea_res = all_gsea_results[col_id]
            fdr = gsea_res[gsea_res['Term']==terms]['FDR q-val'].values
            if len(fdr)==0:
                fdr = 1
            else:
                if len(fdr)>1:
                    AssertionError
                fdr = fdr[0]
            tmp.append(fdr)
        plot_gsea_data.append(tmp)

    plot_df = pd.DataFrame(data = np.concatenate([np.array(plot_gsea_data), np.array(geneset_metadata)[:, None]], axis=1))

    go_res = np.array(plot_df[plot_df[8]=='GO_Biological_Process_2023'].T.iloc[:-1].T.values, dtype=np.float32)
    gwas_res = np.array(plot_df[plot_df[8]=='GWAS_Catalog_2025'].T.iloc[:-1].T.values, dtype=np.float32)
    corum_res = np.array(plot_df[plot_df[8]=='CORUM'].T.iloc[:-1].T.values, dtype=np.float32)
    print('Shape of GO, GWAS, CORUM: ', go_res.shape, gwas_res.shape, corum_res.shape)

    plt.rc('font', size=6)
    plt.figure(figsize=(12,16))
    clustermap_df = pd.DataFrame(-np.log10(go_res+1e-10), index=np.array(all_enriched_terms)[plot_df[8]=='GO_Biological_Process_2023'].tolist(), columns=column_names)
    #clustermap_df = clustermap_df.drop('MsigDB_StringDB', axis=1)
    clustermap_df = clustermap_df.drop(clustermap_df[clustermap_df.T.sum(axis=0)==0].index.tolist(), axis=0)
    if clustermap_df.shape[0] != 0:
        cluster_result = sns.clustermap(clustermap_df, col_cluster=True, vmin=0, vmax=5, cmap='Reds')
        gobp_col_indices = cluster_result.dendrogram_col.reordered_ind
        gobp_row_indices = cluster_result.dendrogram_row.reordered_ind
        save_df_gobp = clustermap_df.iloc[gobp_row_indices, gobp_col_indices]
    else:
        save_df_gobp = None
    

    plt.rc('font', size=6)
    plt.figure(figsize=(6, 4))

    col_indices = cluster_result.dendrogram_col.reordered_ind
    clustermap_df = pd.DataFrame(-np.log10(corum_res+1e-10), index=np.array(all_enriched_terms)[plot_df[8]=='CORUM'].tolist(), columns=column_names)
    clustermap_df = clustermap_df.drop(clustermap_df[clustermap_df.T.sum(axis=0)==0].index.tolist(), axis=0)
    reordered_cols = np.array(clustermap_df.columns.tolist())[col_indices]
    clustermap_df = clustermap_df[reordered_cols]
    if clustermap_df.shape[0] != 0:
        corum_result = sns.clustermap(clustermap_df, col_cluster=False, vmin=0, vmax=8, cmap='Reds')
        corum_row_indices = corum_result.dendrogram_row.reordered_ind
        save_df_corum = clustermap_df.iloc[corum_row_indices]
    else:
        save_df_corum = None

    plt.rc('font', size=6)
    plt.figure(figsize=(6, 4))

    col_indices = cluster_result.dendrogram_col.reordered_ind
    clustermap_df = pd.DataFrame(-np.log10(gwas_res+1e-10), index=np.array(all_enriched_terms)[plot_df[8]=='GWAS_Catalog_2025'].tolist(), columns=column_names)
    clustermap_df = clustermap_df.drop(clustermap_df[clustermap_df.T.sum(axis=0)==0].index.tolist(), axis=0)
    reordered_cols = np.array(clustermap_df.columns.tolist())[col_indices]
    clustermap_df = clustermap_df[reordered_cols]
    if clustermap_df.shape[0] != 0:
        gwas_result = sns.clustermap(clustermap_df, col_cluster=False, vmin=0, vmax=8, cmap='Reds')
        gwas_row_indices = gwas_result.dendrogram_row.reordered_ind
        save_df_gwas = clustermap_df.iloc[gwas_row_indices]
    else:
        save_df_gwas = None

    sns.heatmap(pd.concat([save_df_gobp, save_df_corum, save_df_gwas], axis=0), cmap='vlag', vmax=10, vmin=0, center=0)
    pd.concat([save_df_gobp, save_df_corum, save_df_gwas], axis=0).to_csv(f'gsea_aggregated_{perturb_seq_name}_mrnaspace.csv')

for perturb_seq_name in ['K562_GWPS', 'K562_Essential','RPE1_Essential','HepG2_Essential','Jurkat_Essential','RPE1_Southard']:
    prior_names = ['Marbach', 'MsigDB', 'DepMap', 'StringDB', 'Periscope', 'Untangled', 'scGPT', 'genePT']
    column_names = [x+'_'+perturb_seq_name for x in prior_names]
    print(perturb_seq_name)

    all_enriched_terms = []
    all_gsea_results = []

    for col in prior_names:
        
        tmp = pd.read_csv(f'/home/jhpark/page_reproducibility/03_prior_perturbseq_compare/intermediate/pertspace_{perturb_seq_name}_{col}.csv')
        tmp = tmp[tmp["FDR q-val"]<0.001].sort_values("NES")[::-1]
        all_gsea_results.append(tmp)
        #sns.barplot(tmp, y="Term", x="NES", color="black", orient="y")
        #plt.grid(False)
        all_enriched_terms += tmp['Term'].tolist()

    all_enriched_terms = list(set(all_enriched_terms))

    print('No. of enriched terms:', len(all_enriched_terms))

    plot_gsea_data = []
    geneset_metadata = []
    for terms in all_enriched_terms:
        geneset_metadata.append(terms.split('__')[0])
        tmp = []
        for col_id in range(8):
            gsea_res = all_gsea_results[col_id]
            fdr = gsea_res[gsea_res['Term']==terms]['FDR q-val'].values
            if len(fdr)==0:
                fdr = 1
            else:
                if len(fdr)>1:
                    AssertionError
                fdr = fdr[0]
            tmp.append(fdr)
        plot_gsea_data.append(tmp)

    plot_df = pd.DataFrame(data = np.concatenate([np.array(plot_gsea_data), np.array(geneset_metadata)[:, None]], axis=1))

    go_res = np.array(plot_df[plot_df[8]=='GO_Biological_Process_2023'].T.iloc[:-1].T.values, dtype=np.float32)
    gwas_res = np.array(plot_df[plot_df[8]=='GWAS_Catalog_2025'].T.iloc[:-1].T.values, dtype=np.float32)
    corum_res = np.array(plot_df[plot_df[8]=='CORUM'].T.iloc[:-1].T.values, dtype=np.float32)
    print('Shape of GO, GWAS, CORUM: ', go_res.shape, gwas_res.shape, corum_res.shape)

    plt.rc('font', size=6)
    plt.figure(figsize=(12,16))
    clustermap_df = pd.DataFrame(-np.log10(go_res+1e-10), index=np.array(all_enriched_terms)[plot_df[8]=='GO_Biological_Process_2023'].tolist(), columns=column_names)
    #clustermap_df = clustermap_df.drop('MsigDB_StringDB', axis=1)
    clustermap_df = clustermap_df.drop(clustermap_df[clustermap_df.T.sum(axis=0)==0].index.tolist(), axis=0)
    if clustermap_df.shape[0] != 0:
        cluster_result = sns.clustermap(clustermap_df, col_cluster=True, vmin=0, vmax=5, cmap='Reds')
        gobp_col_indices = cluster_result.dendrogram_col.reordered_ind
        gobp_row_indices = cluster_result.dendrogram_row.reordered_ind
        save_df_gobp = clustermap_df.iloc[gobp_row_indices, gobp_col_indices]
    else:
        save_df_gobp = None
    

    plt.rc('font', size=6)
    plt.figure(figsize=(6, 4))

    col_indices = cluster_result.dendrogram_col.reordered_ind
    clustermap_df = pd.DataFrame(-np.log10(corum_res+1e-10), index=np.array(all_enriched_terms)[plot_df[8]=='CORUM'].tolist(), columns=column_names)
    clustermap_df = clustermap_df.drop(clustermap_df[clustermap_df.T.sum(axis=0)==0].index.tolist(), axis=0)
    reordered_cols = np.array(clustermap_df.columns.tolist())[col_indices]
    clustermap_df = clustermap_df[reordered_cols]
    if clustermap_df.shape[0] != 0:
        corum_result = sns.clustermap(clustermap_df, col_cluster=False, vmin=0, vmax=8, cmap='Reds')
        corum_row_indices = corum_result.dendrogram_row.reordered_ind
        save_df_corum = clustermap_df.iloc[corum_row_indices]
    else:
        save_df_corum = None
    

    plt.rc('font', size=6)
    plt.figure(figsize=(6, 4))

    col_indices = cluster_result.dendrogram_col.reordered_ind
    clustermap_df = pd.DataFrame(-np.log10(gwas_res+1e-10), index=np.array(all_enriched_terms)[plot_df[8]=='GWAS_Catalog_2025'].tolist(), columns=column_names)
    clustermap_df = clustermap_df.drop(clustermap_df[clustermap_df.T.sum(axis=0)==0].index.tolist(), axis=0)
    reordered_cols = np.array(clustermap_df.columns.tolist())[col_indices]
    clustermap_df = clustermap_df[reordered_cols]
    if clustermap_df.shape[0] != 0:
        gwas_result = sns.clustermap(clustermap_df, col_cluster=False, vmin=0, vmax=8, cmap='Reds')
        gwas_row_indices = gwas_result.dendrogram_row.reordered_ind
        save_df_gwas = clustermap_df.iloc[gwas_row_indices]
    else:
        save_df_gwas = None

    try:
        df_to_save = pd.concat([save_df_gobp, save_df_corum, save_df_gwas], axis=0)
        sns.heatmap(df_to_save, cmap='vlag', vmax=10, vmin=0, center=0)
        pd.concat([save_df_gobp, save_df_corum, save_df_gwas], axis=0).to_csv(f'/home/jhpark/page_reproducibility/03_prior_perturbseq_compare/intermediate/gsea_aggregated_{perturb_seq_name}_pertspace.csv')
    except:
        print("Only GO Terms Enriched")
        save_df_gobp.to_csv(f'/home/jhpark/page_reproducibility/03_prior_perturbseq_compare/intermediate/gsea_aggregated_{perturb_seq_name}_pertspace.csv')

### Cofunc

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import warnings
warnings.filterwarnings("ignore")
import io
df = pd.read_csv('/home/jhpark/page_reproducibility/03_prior_perturbseq_compare/intermediate/gsea_aggregated_K562_GWPS_pertspace.csv', index_col=0).clip(lower=0, upper=10)
print(f"Loaded: {df.shape[0]} rows × {df.shape[1]} cols")

# Clusters  (all 85 rows)
clusters = {

    "C1": {
        "color": "#1A1A2E",   # near-black navy
        "terms": [
            "GO_Biological_Process_2023__Translation (GO:0006412)",
            "GO_Biological_Process_2023__Cytoplasmic Translation (GO:0002181)",
            "GO_Biological_Process_2023__Cytoplasmic Translational Initiation (GO:0002183)",
            "GO_Biological_Process_2023__Macromolecule Biosynthetic Process (GO:0009059)",
            "GO_Biological_Process_2023__Ribosomal Large Subunit Biogenesis (GO:0042273)",
            "GO_Biological_Process_2023__Ribonucleoprotein Complex Biogenesis (GO:0022613)",
            "GO_Biological_Process_2023__rRNA Metabolic Process (GO:0016072)",
            "GO_Biological_Process_2023__Peptide Biosynthetic Process (GO:0043043)",
            "GO_Biological_Process_2023__ncRNA Processing (GO:0034470)",
            "GO_Biological_Process_2023__rRNA Processing (GO:0006364)",
            "GO_Biological_Process_2023__Ribosome Assembly (GO:0042255)",
            "GO_Biological_Process_2023__Ribosomal Large Subunit Assembly (GO:0000027)",
            "GO_Biological_Process_2023__Gene Expression (GO:0010467)",
            "GO_Biological_Process_2023__Maturation Of LSU-rRNA From Tricistronic rRNA Transcript (GO:0000463)",
            "GO_Biological_Process_2023__Maturation Of LSU-rRNA From Tricistronic rRNA Transcript (SSU-rRNA, 5.8S rRNA, LSU-rRNA) (GO:0000463)",
            "GO_Biological_Process_2023__Endonucleolytic Cleavage Of Tricistronic rRNA Transcript (GO:0000479)",
            "GO_Biological_Process_2023__Endonucleolytic Cleavage Of Tricistronic rRNA Transcript (SSU-rRNA, 5.8S rRNA, LSU-rRNA) (GO:0000479)",
            "GO_Biological_Process_2023__Maturation Of 5.8S rRNA From Tricistronic rRNA Transcript (GO:0000466)",
            "GO_Biological_Process_2023__Maturation Of 5.8S rRNA From Tricistronic rRNA Transcript (SSU-rRNA, 5.8S rRNA, LSU-rRNA) (GO:0000466)",
            "GO_Biological_Process_2023__Maturation Of 5.8S rRNA (GO:0000460)",
            "GO_Biological_Process_2023__Maturation Of LSU-rRNA (GO:0000470)",
            "GO_Biological_Process_2023__Maturation Of SSU-rRNA (GO:0030490)",
            "GO_Biological_Process_2023__Ribosomal Small Subunit Biogenesis (GO:0042274)",
            "GO_Biological_Process_2023__Ribosome Biogenesis (GO:0042254)",
            "GO_Biological_Process_2023__tRNA Aminoacylation For Protein Translation (GO:0006418)",
            "GO_Biological_Process_2023__Ribosomal Small Subunit Assembly (GO:0000028)",
            "CORUM__Parvulin-associated pre-rRNP complex (mouse)",
            "CORUM__40S ribosomal subunit cytoplasmic (human)",
            "CORUM__40S ribosomal subunit, cytoplasmic (human)",
            "CORUM__60S ribosomal subunit, cytoplasmic (human)",
            "CORUM__60S ribosomal subunit cytoplasmic (human)",
            "CORUM__Ribosome cytoplasmic (human)",
            "CORUM__Ribosome, cytoplasmic (human)",
            "CORUM__TRBP containing complex (human)",
            "CORUM__TRBP containing complex (DICER, RPL7A, EIF6, MOV10 and subunits of the 60S ribosomal particle) (human)",
            "CORUM__Nop56p-associated pre-rRNA complex (human)",
            
        ],
    },

    "C2": {
        "color": "#C0392B",
        "terms": [
            "GO_Biological_Process_2023__Mitochondrial Gene Expression (GO:0140053)",
            "GO_Biological_Process_2023__Mitochondrial Translation (GO:0032543)",
            "GO_Biological_Process_2023__Maturation Of SSU-rRNA From Tricistronic rRNA Transcript (GO:0000462)",
            "GO_Biological_Process_2023__Maturation Of SSU-rRNA From Tricistronic rRNA Transcript (SSU-rRNA, 5.8S rRNA, LSU-rRNA) (GO:0000462)",
            "GO_Biological_Process_2023__Aerobic Electron Transport Chain (GO:0019646)",
            "GO_Biological_Process_2023__Aerobic Respiration (GO:0009060)",
            "GO_Biological_Process_2023__Mitochondrial Electron Transport NADH To Ubiquinone (GO:0006120)",
            "GO_Biological_Process_2023__Mitochondrial Electron Transport, NADH To Ubiquinone (GO:0006120)",
            "GO_Biological_Process_2023__Mitochondrial ATP Synthesis Coupled Electron Transport (GO:0042775)",
            "GO_Biological_Process_2023__Regulation Of Mitochondrial Translation (GO:0070129)",
            "GO_Biological_Process_2023__Positive Regulation Of Mitochondrial Translation (GO:0070131)",
            "GO_Biological_Process_2023__Oxidative Phosphorylation (GO:0006119)",
            "GO_Biological_Process_2023__Proton Motive Force-Driven ATP Synthesis (GO:0015986)",
            "GO_Biological_Process_2023__Proton Motive Force-Driven Mitochondrial ATP Synthesis (GO:0042776)",
            "CORUM__39S ribosomal subunit mitochondrial (human)",
            "CORUM__55S ribosome mitochondrial (human)",
            "CORUM__28S ribosomal subunit mitochondrial (human)",
            "CORUM__39S ribosomal subunit, mitochondrial (human)",
            "CORUM__55S ribosome, mitochondrial (human)",
            "CORUM__28S ribosomal subunit, mitochondrial (human)",
            "CORUM__Respiratory chain complex I mitochondrial (mouse)",
            "CORUM__Respiratory chain complex I, mitochondrial (mouse)",
        ],
    },

    "C3": {
        "color": "#8E44AD",
        "terms": [
            "CORUM__26S proteasome (human)",
            "CORUM__PA700-20S-PA28 complex (human)",
            "CORUM__PA28-20S proteasome (human)",
            "CORUM__PA28gamma-20S proteasome (human)",
            "CORUM__20S proteasome (human)",
        ],
    },

    "C4": {
        "color": "#2471A3",
        "terms": [
            "GO_Biological_Process_2023__Transcription Initiation At RNA Polymerase II Promoter (GO:0006367)",
            "GO_Biological_Process_2023__Transcription Preinitiation Complex Assembly (GO:0070897)",
            "GO_Biological_Process_2023__Positive Regulation Of DNA-templated Transcription Initiation (GO:2000144)",
            "GO_Biological_Process_2023__Regulation Of Transcription Initiation By RNA Pol II (GO:0060260)",
            "GO_Biological_Process_2023__Regulation Of Transcription Initiation By RNA Polymerase II (GO:0060260)",
            "GO_Biological_Process_2023__RNA Polymerase II Preinitiation Complex Assembly (GO:0051123)",
            "GO_Biological_Process_2023__Positive Regulation Of Transcription Initiation By RNA Pol II (GO:0060261)",
            "GO_Biological_Process_2023__Positive Regulation Of Transcription Initiation By RNA Polymerase II (GO:0060261)",
            "GO_Biological_Process_2023__Positive Regulation Of Transcription Elongation By RNA Pol II (GO:0032968)",
            "GO_Biological_Process_2023__Positive Regulation Of Transcription Elongation By RNA Polymerase II (GO:0032968)",
            "GO_Biological_Process_2023__Positive Regulation Of DNA-templated Transcription Elongation (GO:0032786)",
            "GO_Biological_Process_2023__Positive Regulation Of DNA-templated Transcription, Elongation (GO:0032786)",
            "CORUM__Mediator complex (human)",
            "CORUM__RNA polymerase II (RNAPII) (human)",
            "CORUM__BRCA1-RNA polymerase II complex (human)",
            "CORUM__RNA polymerase II holoenzyme complex (human)",
            "CORUM__ARC complex (human)",
            "CORUM__SMCC complex (human)",
            "CORUM__TRAP complex (human)",
            "CORUM__PC2 complex (human)",
            "CORUM__ARC92-Mediator complex (human)",
        ],
    },


    "C5": {
        "color": "#16A085",
        "terms": [
            "GO_Biological_Process_2023__DNA Replication Initiation (GO:0006270)",
            "GO_Biological_Process_2023__DNA-templated DNA Replication (GO:0006261)",
            "GO_Biological_Process_2023__Positive Regulation Of DNA Biosynthetic Process (GO:2000573)",
            "GO_Biological_Process_2023__DNA Unwinding Involved In DNA Replication (GO:0006268)",
            "GO_Biological_Process_2023__DNA Strand Elongation Involved In DNA Replication (GO:0006271)",
            "GO_Biological_Process_2023__Regulation Of DNA-templated DNA Replication (GO:0090329)",
            "GO_Biological_Process_2023__Regulation Of DNA-templated DNA Replication Initiation (GO:0030174)",
            "GO_Biological_Process_2023__protein-RNA Complex Assembly (GO:0022618)",
            "GO_Biological_Process_2023__U2-type Prespliceosome Assembly (GO:1903241)",
            "CORUM__Spliceosome (human)",
            "CORUM__17S U2 snRNP (human)",
        ],
    },

    "C6": {
        "color": "#27AE60",
        "terms": [
            "GO_Biological_Process_2023__rRNA Catabolic Process (GO:0016075)",
            "GO_Biological_Process_2023__Nucleobase-Containing Compound Catabolic Process (GO:0034655)",
            "GO_Biological_Process_2023__Nuclear RNA Surveillance (GO:0071027)",
            "GO_Biological_Process_2023__Nuclear-Transcribed mRNA Catabolic Process Exonucleolytic (GO:0000291)",
            "GO_Biological_Process_2023__Nuclear-Transcribed mRNA Catabolic Process, Exonucleolytic (GO:0000291)",
            "CORUM__Exosome (human)",
        ],
    },

    "C7": {
        "color": "#D35400", # not assigned
        "terms": [
            "GO_Biological_Process_2023__Negative Regulation Of Ubiquitin-Protein Transferase Activity (GO:0051444)",
            "GO_Biological_Process_2023__Regulation Of Signal Transduction By P53 Class Mediator (GO:1901796)",
        ],
    },
}

# Verify all assigned
assigned = [t for cd in clusters.values() for t in cd["terms"]]
missing = [r for r in df.index if r not in assigned]
print(f"Unassigned rows: {len(missing)}")
for m in missing:
    print(" ", m)

col_order = [
    "scGPT_K562_GWPS",
    "genePT_K562_GWPS",
    "MsigDB_K562_GWPS",
    "StringDB_K562_GWPS",
    "DepMap_K562_GWPS",
    "Periscope_K562_GWPS",
    "Untangled_K562_GWPS",
    "Marbach_K562_GWPS",
]

# build ordered matrix
import re
all_terms, clabels, ccolors = [], [], []
for cname, cdata in clusters.items():
    valid = [t for t in cdata["terms"] if t in df.index]
    all_terms += valid
    clabels   += [cname] * len(valid)
    ccolors   += [cdata["color"]] * len(valid)

valid_cols = [c for c in col_order if c in df.columns]
plot_df = df.loc[all_terms, valid_cols].fillna(0).clip(upper=10)
n_rows, n_cols = plot_df.shape
print(f"Final matrix: {n_rows}×{n_cols}")

def shorten(s):
    s = re.sub(r"GO_Biological_Process_2023__", "GO: ", s)
    s = re.sub(r"CORUM__", "C: ", s)
    s = re.sub(r"\s*\(GO:[0-9]+\)\s*$", "", s)
    s = re.sub(r"\s*\(human\)$", "", s)
    s = re.sub(r"\s*\(mouse\)$", "", s)
    if len(s) > 65: s = s[:62] + "…"
    return s

row_labels = [shorten(t) for t in all_terms]

# Short col labels
col_labels = [c.replace("_K562_GWPS", "").replace("_", "\n") for c in valid_cols]

# plotter

ROW_H, COL_W = 0.25, 0.75
LBAR_W, LEFT_PAD = 1.2, 0.08
TOP_H, BOT_H    = 1.5, 0.9

fig_w = LEFT_PAD + LBAR_W + n_cols * COL_W
fig_h = TOP_H + n_rows * ROW_H + BOT_H

fig = plt.figure(figsize=(fig_w, fig_h))

# axes in figure-fraction coords
ax_w = n_cols * COL_W / fig_w
ax_h = n_rows * ROW_H / fig_h
l    = (LEFT_PAD + LBAR_W) / fig_w
b    = BOT_H / fig_h

ax_heat = fig.add_axes([l, b, ax_w, ax_h])
#ax_cbar = fig.add_axes([(LEFT_PAD / fig_w), b, LBAR_W / fig_w * 0.92, ax_h])

#heatmap plotter
cmap = matplotlib.colormaps.get_cmap("Blues")
mat  = plot_df.values
im   = ax_heat.imshow(mat, aspect="auto", cmap=cmap,
                      vmin=0, vmax=10, interpolation="nearest")

ax_heat.set_xlim(-0.5, n_cols - 0.5)
ax_heat.set_ylim(n_rows - 0.5, -0.5)
ax_heat.set_xticks([])
ax_heat.set_yticks([])
#ax_heat.set_yticks(range(n_rows))
#ax_heat.set_yticklabels(row_labels, fontsize=4.8, family="monospace")
ax_heat.tick_params(axis="y", length=0, pad=2)
ax_heat.spines[:].set_visible(False)

for i in range(n_rows + 1):
    ax_heat.axhline(i - 0.5, color="white", lw=0.3)
for j in range(n_cols + 1):
    ax_heat.axvline(j - 0.5, color="white", lw=0.5)

# cluster separator
seen_cl = []
for i, cl in enumerate(clabels):
    if cl not in seen_cl:
        if i > 0:
            ax_heat.axhline(i - 0.5, color="#333", lw=1.2)
        seen_cl.append(cl)

cb_ax = fig.add_axes([LEFT_PAD / fig_w,
                      (BOT_H * 0.15) / fig_h,
                      (LBAR_W + n_cols * COL_W) / fig_w * 0.45,
                      (BOT_H * 0.20) / fig_h])
cb = fig.colorbar(im, cax=cb_ax, orientation="horizontal")
cb.set_label("Enrichment  (−log P)", fontsize=7)
cb.ax.tick_params(labelsize=6.5)
cb.set_ticks([0, 2, 4, 6, 8, 10])
cb.ax.set_xticklabels(["0", "2", "4", "6", "8", "≥10"])

#plt.savefig('../figure_out/Fig.3E_1.pdf')

Loaded: 85 rows × 8 cols
Unassigned rows: 0
Final matrix: 85×8


In [ ]:
cluster_df = pd.DataFrame([all_terms, clabels], index=['terms', 'cluster']).T
cluster_dict = {}
for i in range(1, 8):
    cluster_dict[f'C{i}']= cluster_df[cluster_df['cluster'] == f'C{i}'].terms.tolist()
import json
#with open('/home/jhpark/page_reproducibility/03_prior_perturbseq_compare/intermediate/priorpert_cofunc_gsea_cluster.json', 'w') as f:
#    json.dump(cluster_dict, f)

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(20, 1))
membership = []
for terms in plot_df.index:
    m = terms.split('_')[0]
    if m == 'GO':
        membership.append(1)
    elif m == 'CORUM':
        membership.append(2)
    elif m == 'GWAS':
        membership.append(3)
membership = np.array(membership)
ax.imshow(np.array([membership, membership, membership]), cmap='Pastel2')
ax.axis('off')
plt.tight_layout()
#plt.savefig('../figure_out/Fig.3E_1_terms.pdf')

### Coreg

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import warnings, io, re
warnings.filterwarnings("ignore")

df = pd.read_csv('/home/jhpark/page_reproducibility/03_prior_perturbseq_compare/intermediate/gsea_aggregated_K562_GWPS_mrnaspace.csv', index_col=0).clip(lower=0, upper=10)
print(f"Loaded: {df.shape[0]} rows × {df.shape[1]} cols")

clusters = {

    "C1": {
        "color": "#E67E22",
        "terms": [
            "GO_Biological_Process_2023__Ribosome Biogenesis (GO:0042254)",
            "GO_Biological_Process_2023__ncRNA Processing (GO:0034470)",
            "GO_Biological_Process_2023__rRNA Processing (GO:0006364)",
            "GO_Biological_Process_2023__rRNA Metabolic Process (GO:0016072)",
            "GO_Biological_Process_2023__Ribonucleoprotein Complex Biogenesis (GO:0022613)",
            "GO_Biological_Process_2023__Ribosomal Small Subunit Biogenesis (GO:0042274)",
            "GO_Biological_Process_2023__Ribosomal Large Subunit Biogenesis (GO:0042273)",
            "GO_Biological_Process_2023__Maturation Of SSU-rRNA (GO:0030490)",
            "GO_Biological_Process_2023__Maturation Of LSU-rRNA (GO:0000470)",
            "GO_Biological_Process_2023__Maturation Of SSU-rRNA From Tricistronic rRNA Transcript (GO:0000462)",
            "GO_Biological_Process_2023__Maturation Of SSU-rRNA From Tricistronic rRNA Transcript (SSU-rRNA, 5.8S rRNA, LSU-rRNA) (GO:0000462)",
            "GO_Biological_Process_2023__Cytoplasmic Translation (GO:0002181)",
            "GO_Biological_Process_2023__Cytoplasmic Translational Initiation (GO:0002183)",
            "GO_Biological_Process_2023__Macromolecule Biosynthetic Process (GO:0009059)",
            "GO_Biological_Process_2023__Peptide Biosynthetic Process (GO:0043043)",
            "GO_Biological_Process_2023__Translation (GO:0006412)",
            "GO_Biological_Process_2023__Gene Expression (GO:0010467)",
            "GO_Biological_Process_2023__Ribosome Assembly (GO:0042255)",
            "GO_Biological_Process_2023__Ribosomal Large Subunit Assembly (GO:0000027)",
            "CORUM__Nop56p-associated pre-rRNA complex (human)",
            "CORUM__60S ribosomal subunit cytoplasmic (human)",
            "CORUM__60S ribosomal subunit, cytoplasmic (human)",
            "CORUM__Ribosome cytoplasmic (human)",
            "CORUM__Ribosome, cytoplasmic (human)",
            "CORUM__40S ribosomal subunit cytoplasmic (human)",
            "CORUM__40S ribosomal subunit, cytoplasmic (human)",
            "CORUM__TRBP containing complex (human)",
            "CORUM__TRBP containing complex (DICER, RPL7A, EIF6, MOV10 and subunits of the 60S ribosomal particle) (human)",
            "CORUM__Parvulin-associated pre-rRNP complex (mouse)",
        ],
    },

    "C2": {
        "color": "#C0392B",
        "terms": [
            "GO_Biological_Process_2023__Mitochondrial Electron Transport NADH To Ubiquinone (GO:0006120)",
            "GO_Biological_Process_2023__Mitochondrial Electron Transport, NADH To Ubiquinone (GO:0006120)",
            "GO_Biological_Process_2023__Mitochondrial ATP Synthesis Coupled Electron Transport (GO:0042775)",
            "GO_Biological_Process_2023__Aerobic Electron Transport Chain (GO:0019646)",
            "GO_Biological_Process_2023__Proton Motive Force-Driven ATP Synthesis (GO:0015986)",
            "GO_Biological_Process_2023__Proton Motive Force-Driven Mitochondrial ATP Synthesis (GO:0042776)",
            "GO_Biological_Process_2023__Mitochondrial Translation (GO:0032543)",
            "GO_Biological_Process_2023__Cellular Respiration (GO:0045333)",
            "GO_Biological_Process_2023__Aerobic Respiration (GO:0009060)",
            "GO_Biological_Process_2023__Oxidative Phosphorylation (GO:0006119)",
            "GO_Biological_Process_2023__Energy Derivation By Oxidation Of Organic Compounds (GO:0015980)",
            "GO_Biological_Process_2023__NADH Dehydrogenase Complex Assembly (GO:0010257)",
            "GO_Biological_Process_2023__Mitochondrial Respiratory Chain Complex I Assembly (GO:0032981)",
            "GO_Biological_Process_2023__Mitochondrial Respiratory Chain Complex Assembly (GO:0033108)",
            "GO_Biological_Process_2023__Mitochondrial Gene Expression (GO:0140053)",
            "CORUM__Respiratory chain complex I mitochondrial (mouse)",
            "CORUM__Respiratory chain complex I, mitochondrial (mouse)",
            "CORUM__55S ribosome mitochondrial (human)",
            "CORUM__55S ribosome, mitochondrial (human)",
            "CORUM__39S ribosomal subunit mitochondrial (human)",
            "CORUM__39S ribosomal subunit, mitochondrial (human)",
            "CORUM__28S ribosomal subunit mitochondrial (human)",
            "CORUM__28S ribosomal subunit, mitochondrial (human)",
        ],
    },
    
    "C3": {
        "color": "#8E44AD",
        "terms": [
            "CORUM__PA700-20S-PA28 complex (human)",
            "CORUM__26S proteasome (human)",
            "CORUM__20S proteasome (human)",
            "CORUM__PA28-20S proteasome (human)",
            "CORUM__PA28gamma-20S proteasome (human)",
        ],
    },

    "C4": {
        "color": "#2471A3",
        "terms": [
            "GO_Biological_Process_2023__RNA Splicing Via Transesterification Reactions (GO:0000375)",
            "GO_Biological_Process_2023__RNA Splicing, Via Transesterification Reactions (GO:0000375)",
            "GO_Biological_Process_2023__mRNA Splicing, Via Spliceosome (GO:0000398)",
            "GO_Biological_Process_2023__RNA Splicing, Via Transesterification Reactions With Bulged Adenosine As Nucleophile (GO:0000377)",
            "GO_Biological_Process_2023__Positive Regulation Of mRNA Splicing, Via Spliceosome (GO:0048026)",
            "GO_Biological_Process_2023__RNA Splicing (GO:0008380)",
            "GO_Biological_Process_2023__Positive Regulation Of Transcription By RNA Polymerase I (GO:0045943)",
            "GO_Biological_Process_2023__mRNA Splicing Via Spliceosome (GO:0000398)",
            "GO_Biological_Process_2023__mRNA Processing (GO:0006397)",
            "GO_Biological_Process_2023__RNA Splicing Via Transesterification With Bulged Adenosine (GO:0000377)",
            "GO_Biological_Process_2023__Spliceosomal snRNP Assembly (GO:0000387)",
            "GO_Biological_Process_2023__protein-RNA Complex Assembly (GO:0022618)",
            "GO_Biological_Process_2023__RNA Processing (GO:0006396)",
            "GO_Biological_Process_2023__Spliceosomal Complex Assembly (GO:0000245)",
            "GO_Biological_Process_2023__Positive Regulation Of mRNA Processing (GO:0050685)",
            "GO_Biological_Process_2023__Positive Regulation Of mRNA Splicing Via Spliceosome (GO:0048026)",
            "GO_Biological_Process_2023__Nuclear Transport (GO:0051169)",
            "GO_Biological_Process_2023__Nucleocytoplasmic Transport (GO:0006913)",
            "GO_Biological_Process_2023__U2-type Prespliceosome Assembly (GO:1903241)",
            "CORUM__Spliceosome (human)",
            "CORUM__SMN complex (human)",
            "CORUM__C complex spliceosome (human)",
        ],
    },

    "C5": {
        "color": "#117A65",
        "terms": [
            "GO_Biological_Process_2023__Protein Insertion Into Mitochondrial Membrane (GO:0051204)",
            "GO_Biological_Process_2023__Protein Targeting To Mitochondrion (GO:0006626)",
            "GO_Biological_Process_2023__Protein Insertion Into Mitochondrial Outer Membrane (GO:0045040)",
            "GO_Biological_Process_2023__Outer Mitochondrial Membrane Organization (GO:0007008)",
        ],
    },


    "C6": {
        "color": "#566573",
        "terms": [
            "GO_Biological_Process_2023__Microtubule Cytoskeleton Organization Involved In Mitosis (GO:1902850)",
            "GO_Biological_Process_2023__Chromosome Condensation (GO:0030261)",
            "GO_Biological_Process_2023__Mitotic Cytokinesis (GO:0000281)",
            "GO_Biological_Process_2023__Regulation Of Ubiquitin Protein Ligase Activity (GO:1904666)",
            "GO_Biological_Process_2023__Protein O-linked Glycosylation (GO:0006493)",
            "GO_Biological_Process_2023__Positive Regulation Of Mitotic Sister Chromatid Separation (GO:1901970)",
            "GO_Biological_Process_2023__Negative Regulation Of Ubiquitin-Protein Transferase Activity (GO:0051444)",
            "GO_Biological_Process_2023__Negative Regulation Of Mitotic Metaphase/Anaphase Transition (GO:0045841)",
            "GO_Biological_Process_2023__Mitotic Spindle Organization (GO:0007052)",
            "GO_Biological_Process_2023__Mitotic Spindle Checkpoint Signaling (GO:0071174)",
            "GO_Biological_Process_2023__Spindle Assembly Checkpoint Signaling (GO:0071173)",
            "GO_Biological_Process_2023__Mitotic Spindle Assembly Checkpoint Signaling (GO:0007094)",
        ],
    },

    "C7": {
        "color": "#AAB7B8",
        "terms": [
            "GO_Biological_Process_2023__Positive Regulation Of Signal Transduction By P53 Class Mediator (GO:1901798)",
            "GO_Biological_Process_2023__mRNA Export From Nucleus (GO:0006406)",
        ],
    },
}

assigned = [t for cd in clusters.values() for t in cd["terms"]]
missing  = [r for r in df.index if r not in assigned]
print(f"Unassigned rows: {len(missing)}")
for m in missing: print(" ", m)

col_order = [
    "scGPT_K562_GWPS",
    "genePT_K562_GWPS",
    "MsigDB_K562_GWPS",
    "StringDB_K562_GWPS",
    "DepMap_K562_GWPS",
    "Periscope_K562_GWPS",
    "Untangled_K562_GWPS",
    "Marbach_K562_GWPS",
]

all_terms, clabels, ccolors = [], [], []
for cname, cdata in clusters.items():
    valid = [t for t in cdata["terms"] if t in df.index]
    all_terms += valid
    clabels   += [cname] * len(valid)
    ccolors   += [cdata["color"]] * len(valid)

valid_cols = [c for c in col_order if c in df.columns]
plot_df    = df.loc[all_terms, valid_cols].fillna(0).clip(lower=0, upper=10)
n_rows, n_cols = plot_df.shape
print(f"Final matrix: {n_rows} × {n_cols}")

def shorten(s):
    s = re.sub(r"GO_Biological_Process_2023__", "GO: ", s)
    s = re.sub(r"CORUM__", "C: ", s)
    s = re.sub(r"\s*\(GO:[0-9]+\)\s*$", "", s)
    s = re.sub(r"\s*\(human\)$", "", s)
    s = re.sub(r"\s*\(mouse\)$", "", s)
    if len(s) > 64: s = s[:61] + "…"
    return s

row_labels = [shorten(t) for t in all_terms]
col_labels = [c.replace("_K562_GWPS", "").replace("_", "\n") for c in valid_cols]

ROW_H, COL_W = 0.25, 0.75
LBAR_W, LEFT_PAD = 1.2, 0.08
TOP_H, BOT_H    = 1.5, 0.9

fig_w = LEFT_PAD + LBAR_W + n_cols * COL_W
fig_h = TOP_H + n_rows * ROW_H + BOT_H

fig = plt.figure(figsize=(fig_w, fig_h))

ax_w = n_cols * COL_W / fig_w
ax_h = n_rows * ROW_H / fig_h
l    = (LEFT_PAD + LBAR_W) / fig_w
b    = BOT_H / fig_h

ax_heat = fig.add_axes([l, b, ax_w, ax_h])
#ax_cbar = fig.add_axes([LEFT_PAD / fig_w, b, LBAR_W / fig_w * 0.92, ax_h])

cmap = matplotlib.colormaps.get_cmap("Blues")
#cmap.set_under("#F7F7F7")
im = ax_heat.imshow(plot_df.values, aspect="auto", cmap=cmap,
                    vmin=0.01, vmax=10, interpolation="nearest")

ax_heat.set_xlim(-0.5, n_cols - 0.5)
ax_heat.set_ylim(n_rows - 0.5, -0.5)
ax_heat.set_xticks([])
ax_heat.set_yticks([])
#ax_heat.set_yticks(range(n_rows))
#ax_heat.set_yticklabels(row_labels, fontsize=4.8, family="monospace")
ax_heat.tick_params(axis="y", length=0, pad=2)
ax_heat.spines[:].set_visible(False)
for i in range(n_rows + 1): ax_heat.axhline(i - 0.5, color="white", lw=0.3)
for j in range(n_cols + 1): ax_heat.axvline(j - 0.5, color="white", lw=0.5)

seen_cl = []
for i, cl in enumerate(clabels):
    if cl not in seen_cl:
        if i > 0: ax_heat.axhline(i - 0.5, color="#333", lw=1.2)
        seen_cl.append(cl)

#plt.savefig('../figure_out/Fig.3E_2.pdf')

Loaded: 82 rows × 8 cols
Unassigned rows: 0
Final matrix: 82 × 8


In [ ]:
cluster_df = pd.DataFrame([all_terms, clabels], index=['terms', 'cluster']).T
cluster_dict = {}
for i in range(1, 8):
    cluster_dict[f'C{i}']= cluster_df[cluster_df['cluster'] == f'C{i}'].terms.tolist()
import json
#with open('/home/jhpark/page_reproducibility/03_prior_perturbseq_compare/intermediate/priorpert_coreg_gsea_cluster.json', 'w') as f:
#    json.dump(cluster_dict, f)

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(20, 1))
membership = []
for terms in plot_df.index:
    m = terms.split('_')[0]
    if m == 'GO':
        membership.append(1)
    elif m == 'CORUM':
        membership.append(2)
    elif m == 'GWAS':
        membership.append(3)
membership = np.array(membership)
ax.imshow(np.array([membership, membership, membership]), cmap='Pastel2')
ax.axis('off')
plt.tight_layout()
#plt.savefig('../figure_out/Fig.3E_2_terms.pdf')